# 4.0 — Strategy Evaluation & Simulation

Deep-dive into strategy quality: external validity, portfolio simulation,
stable-feature iteration, and factor-neutral attribution.

Loads model outputs from `models/` (produced by notebook 3.0 or `make all`).

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from stock_data.config import INITIAL_CAPITAL
from stock_data.evaluation import (
    assess_external_validity,
    simulate_portfolio,
    print_simulation_summary,
    run_iteration_analysis,
)
from stock_data.plots import plot_simulation

## Load Artifacts

In [ ]:
PROCESSED = Path("data/processed")
INTERIM = Path("data/interim")
MODELS = Path("models")

risk_model_df = pd.read_parquet(PROCESSED / "risk_model_df.parquet")
with open(PROCESSED / "feature_cols.pkl", "rb") as f:
    feature_cols_all = pickle.load(f)
close_prices = pd.read_parquet(INTERIM / "close_prices.parquet")

prod_df = pd.read_parquet(MODELS / "prod_df.parquet")
with open(MODELS / "prod_fi.pkl", "rb") as f:
    prod_fi = pickle.load(f)
with open(MODELS / "weights_history.pkl", "rb") as f:
    prod_weights_history = pickle.load(f)

print(f"Loaded {len(prod_df)} production quarters, "
      f"{risk_model_df.shape[0]} rows, {len(feature_cols_all)} features")

## External Validity Assessment

In [ ]:
assess_external_validity(
    prod_df, prod_fi, prod_weights_history,
    risk_model_df, feature_cols_all, close_prices,
)

## Portfolio Simulation

In [ ]:
print("=" * 80)
print(f"PRACTICE SIMULATION: ${INITIAL_CAPITAL:,.0f} PORTFOLIO")
print("=" * 80)

sim_df, mkt_sim, qlog = simulate_portfolio(
    prod_df, prod_weights_history, risk_model_df, close_prices, INITIAL_CAPITAL,
)

if len(sim_df) > 0 and len(mkt_sim) > 0:
    fig = plot_simulation(sim_df, mkt_sim, qlog, INITIAL_CAPITAL)
    fig.savefig("reports/figures/simulation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print_simulation_summary(sim_df, mkt_sim, INITIAL_CAPITAL)

## Iteration: Stable Features & Factor-Neutral Attribution

In [ ]:
run_iteration_analysis(
    prod_df, prod_fi, prod_weights_history,
    risk_model_df, feature_cols_all, close_prices,
)